# Stage 1 - Data audit: Delhi OTD bus GTFS

**Purpose.** Before cleaning or feature engineering, measure what the feed actually contains: table shapes, missing values, duplicates, coordinate ranges, referential integrity and whether times exist at all.

**Data note.** The official full Delhi OTD zip is downloaded through a POST form (name, e-mail, terms acceptance) and its file host was unreachable when this notebook was written. This notebook therefore runs on a public *trimmed* copy of the same OTD data, kept in `data/raw/delhi_gtfs_trimmed/`. It is **real Delhi bus data, not synthetic**, but it is trimmed: see the audit findings for exactly what is missing. The same code runs unchanged on the full zip.

In [1]:
import pathlib
import sys

sys.path.append(str(pathlib.Path.cwd().parent))  # make `src` importable from notebooks/

import matplotlib
matplotlib.use("Agg")  # no display needed; we save figures to disk
import matplotlib.pyplot as plt
import pandas as pd

from src.io import PROJECT_ROOT, load_gtfs
from src.audit import print_report, run_audit

FEED_DIR = PROJECT_ROOT / "data" / "raw" / "delhi_gtfs_trimmed"
feed = load_gtfs(FEED_DIR)

pd.DataFrame({name: {"rows": len(df), "cols": df.shape[1]} for name, df in feed.items()}).T

,rows,cols
agency,2,7
calendar,1,10
routes,2403,6
trips,2403,4
stops,10559,4
stop_times,99286,3


In [2]:
# Stops carry the coordinates every spatial feature depends on, so inspect them directly.
stops = feed["stops"]
print(stops.dtypes)
stops.head()

stop_id       string
stop_name     string
stop_lat     float64
stop_lon     float64
dtype: object


,stop_id,stop_name,stop_lat,stop_lon
0,0,Narela Terminal,28.851958,77.088107
1,1,Police Station Narela,28.853311,77.088548
2,2,Safiyabad Crossing,28.854965,77.089322
3,3,Ramdev Chowk / Pitori Johad,28.857713,77.092846
4,4,Narela A-6 / CPJ College,28.857684,77.097117


In [3]:
# Full audit: shapes, missing values, duplicates, coordinates, integrity, calendar.
report = run_audit(FEED_DIR, label="Delhi OTD (trimmed public copy)")
print_report(report)


FleetSense Stage 1 audit - Delhi OTD (trimmed public copy)
path: /home/ashira/Documents/Projects/Fleetsense/data/raw/delhi_gtfs_trimmed
run : 2026-11-09T06:31:25+00:00   synthetic: False

-- tables --
     table  rows  cols  dup_rows
    agency     2     7         0
  calendar     1    10         0
    routes  2403     6         0
     trips  2403     4         0
     stops 10559     4         0
stop_times 99286     3         0

-- coordinates --
stops=10559  lat=[28.41665, 28.87187]  lon=[76.84277, 77.41455]  outside_envelope=0 (0.0%)

-- integrity --
arrival/departure times present : False
routes=2403  trips/route median=1.0  routes_with_one_trip=2403
stops/trip median=42.0  duplicate (trip,seq)=0  nonmonotonic trips=0

-- findings --
1. stop_times has NO arrival_time/departure_time columns: this feed cannot give schedule delay, dwell time or headway. Get the full official zip.
2. 2403 of 2403 routes have exactly one trip, so this is a trimmed sample, not a working timetable. Recurr

In [4]:
# Visual sanity check: do the stop coordinates look like one coherent city, or are there strays?
lat = pd.to_numeric(stops["stop_lat"], errors="coerce")
lon = pd.to_numeric(stops["stop_lon"], errors="coerce")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(lon, lat, s=3, alpha=0.35, c="#111111", linewidths=0)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"Delhi bus stops in the audit feed (n={len(stops)})")
ax.set_aspect("equal", adjustable="datalim")

fig_path = PROJECT_ROOT / "outputs" / "figures" / "stage1_stop_coverage.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=140, bbox_inches="tight")
print(f"saved {fig_path}")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage1_stop_coverage.png


## How to read this

* If `has_arrival_departure_times` is `False`, then dwell time, headway and schedule delay are **not computable** from this feed. Stage 3 must not pretend otherwise.
* If `routes_with_one_trip` equals the number of routes, the feed is a route catalogue, not a timetable: recurring temporal regimes cannot be learned from it.
* `n_duplicate_coordinate_pairs` tells you how much stop-merging matters before any hex-cell aggregation.
* The saved `outputs/data_audit.json` is the machine-readable version of everything printed above, and is what the app's *Data and Method* page will read later.